# 01 Held-out test of hidden-subtype discovery (69 pools, 8 atlases)
Recomputes the headline comparison of the paper: atlas-equal mean union average precision (AP) and E1 of our method and of Leiden subclustering on the 69 held-out pools, the paired tests, and the learned-scorer and pseudo-label-loop numbers quoted next to them.

Inputs are the release assets only (`pools.tgz`, `predictions.tgz`, extracted under `ASSETS`): frozen cell scores, the 69 pools' visible labels and hidden finer labels, and the atlas of each pool. Nothing is refit. A hidden target is a finer type with at least 10 cells and under 10% of its visible label. E1 is the precision among the top K cells, K the number of target cells, ties split at their expected value.

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
REPO = os.path.abspath(os.environ.get("REPO", ".."))
ASSETS = os.path.abspath(os.environ.get("ASSETS", os.path.join(REPO, "assets")))
sys.path.insert(0, REPO); sys.dont_write_bytecode = True
import json, numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
EXP = f"{REPO}/eval/expected"
from eval import paper as P
pids, man = P.pools(ASSETS)
assert len(pids) == 69 and man.atlas.nunique() == 8
print(len(pids), "pools;", man.atlas.value_counts().to_dict())

69 pools; {'hspc': 21, 'ocular': 14, 'arthritis': 8, 'htapp': 6, 'psccolon': 6, 'cereb': 5, 'hypomap': 5, 'luad': 4}


## 1. Score every method on every pool
Our method (`hiddenrare`), Leiden subclustering (`leiden_sub`), the learned scorer alone (`scorer`) and the three pseudo-label loops.

In [2]:
METH = ["hiddenrare", "leiden_sub", "scorer", "dup_loop", "nodup_loop", "classweight_loop"]
T = P.pool_metrics(ASSETS, {m: f"{ASSETS}/predictions/{m}" for m in METH}, pids)
assert T.groupby("method").pilot_id.nunique().eq(69).all()
print("atlas-equal means")
print(pd.DataFrame({met: P.atlas_equal(P.wide(T, met), man) for met in P.METRICS}).round(4))

atlas-equal means
                  union_AP  E1_union
method                              
classweight_loop    0.0729    0.0819
dup_loop            0.0686    0.0789
hiddenrare          0.3213    0.3227
leiden_sub          0.2204    0.2339
nodup_loop          0.0720    0.0801
scorer              0.2842    0.2920


## 2. Our method against Leiden subclustering
Three tests are reported. The exact atlas-level sign-flip (all 2^8 sign patterns of the eight atlas mean differences) carries the pass rule. The pool-level weighted sign-flip (20,000 draws, seed 0) cannot go below 1/20001, so its value is a bound. The interval is a two-level percentile bootstrap (atlases, then pools within each sampled atlas; 2,000 draws, seed 0); the paper's interval comes from a different random stream, so the check allows 0.005.

In [3]:
A, L = "hiddenrare", "leiden_sub"; res = {}
for met in P.METRICS:
    w = P.wide(T, met); r = P.versus(w, A, L, man)
    res[met] = dict(ours=r["a_mean"], leiden=r["b_mean"], diff=r["diff"], atlases_ahead=f"{r['atlas_ahead']}/{r['n_atlases']}", p_atlas_exact=r["p"], p_pool_flip=r["p_pool_flip"],
                    ci_lo=r["ci"][0], ci_hi=r["ci"][1], loo_min=r["loo"][0], loo_max=r["loo"][1], atlas_sd=r["atlas_sd"], per_atlas=r["per_atlas_diff"])
hp = P.holm({k: v["p_pool_flip"] for k, v in res.items()})
for k in res: res[k]["p_pool_flip_holm"] = hp[k]
print(pd.DataFrame(res).T.drop(columns="per_atlas").round(4).to_string())
ap, e1 = res["union_AP"], res["E1_union"]
assert (round(ap["ours"], 4), round(ap["leiden"], 4), round(ap["diff"], 4)) == (0.3213, 0.2204, 0.1009)
assert ap["atlases_ahead"] == "6/8" and abs(ap["p_atlas_exact"] - 0.03125) < 1e-12
assert ap["p_pool_flip"] <= 5e-5 and round(ap["p_pool_flip_holm"], 4) == 0.0001
assert (round(ap["loo_min"], 3), round(ap["loo_max"], 3), round(ap["atlas_sd"], 3)) == (0.077, 0.122, 0.111)
assert (round(e1["ours"], 4), round(e1["leiden"], 4), round(e1["diff"], 4)) == (0.3227, 0.2339, 0.0887)
assert e1["atlases_ahead"] == "6/8" and round(e1["p_atlas_exact"], 3) == 0.086   # not significant at 0.05
assert abs(ap["ci_lo"] - 0.026) < 0.005 and abs(ap["ci_hi"] - 0.177) < 0.005
exp = json.load(open(f"{EXP}/per_atlas_diff.json"))
assert all(abs(ap["per_atlas"][a] - exp[a]) < 1e-9 for a in exp), "per-atlas differences differ from the paper's figure data"
print("Union AP: ours %.3f, Leiden %.3f, +%.3f (95%% interval %+.3f to %+.3f), ahead in %s atlases, exact atlas-level p = %.5f" % (ap["ours"], ap["leiden"], ap["diff"], ap["ci_lo"], ap["ci_hi"], ap["atlases_ahead"], ap["p_atlas_exact"]))
print("E1: ours %.3f, Leiden %.3f, +%.3f, ahead in %s, exact p = %.4f (not significant at 0.05)" % (e1["ours"], e1["leiden"], e1["diff"], e1["atlases_ahead"], e1["p_atlas_exact"]))

              ours    leiden      diff atlases_ahead p_atlas_exact p_pool_flip     ci_lo     ci_hi   loo_min   loo_max  atlas_sd p_pool_flip_holm
union_AP  0.321311  0.220397  0.100915           6/8       0.03125     0.00005  0.028714  0.180196  0.076977  0.121745   0.11139           0.0001
E1_union  0.322663   0.23392  0.088743           6/8      0.085938     0.00005  0.004224  0.176113   0.06517  0.112616  0.124209           0.0001
Union AP: ours 0.321, Leiden 0.220, +0.101 (95% interval +0.029 to +0.180), ahead in 6/8 atlases, exact atlas-level p = 0.03125
E1: ours 0.323, Leiden 0.234, +0.089, ahead in 6/8, exact p = 0.0859 (not significant at 0.05)


## 3. Per-atlas union AP
The differences are the bars of the per-atlas figure.

In [4]:
w = P.wide(T, "union_AP"); w["atlas"] = man.atlas
pa = w.groupby("atlas")[[A, L]].mean(); pa["diff"] = pa[A] - pa[L]; pa["pools"] = w.groupby("atlas").size()
print(pa.sort_values("diff").round(3))

method     hiddenrare  leiden_sub   diff  pools
atlas                                          
htapp           0.297       0.342 -0.045      6
luad            0.063       0.068 -0.004      4
hypomap         0.146       0.094  0.052      5
psccolon        0.159       0.090  0.069      6
cereb           0.340       0.233  0.107      5
ocular          0.363       0.256  0.107     14
hspc            0.641       0.388  0.253     21
arthritis       0.561       0.292  0.268      8


## 4. Learned scorer alone and the Leiden blend
The blend of the learned scorer with the Leiden score (our method) against the scorer alone, on the same pools.

In [5]:
w = P.wide(T, "union_AP"); ae = P.atlas_equal(w, man)
vs_l = P.versus(w, "scorer", L, man); vs_s = P.versus(w, A, "scorer", man)
print(f"scorer alone {ae['scorer']:.3f}, ahead of Leiden by {vs_l['diff']:+.3f}; blend adds {vs_s['diff']:+.3f} (pool-level p {vs_s['p_pool_flip']:.4f}); share of the gain from the scorer {vs_l['diff'] / ap['diff']:.0%}")
assert round(ae["scorer"], 3) == 0.284 and round(vs_l["diff"], 3) == 0.064 and round(vs_s["diff"], 3) == 0.037 and abs(vs_s["p_pool_flip"] - 11 / 20001) < 1e-12   # 0.00055; the paper prints 0.0006
assert round(vs_l["diff"] / ap["diff"], 2) == 0.63

scorer alone 0.284, ahead of Leiden by +0.064; blend adds +0.037 (pool-level p 0.0005); share of the gain from the scorer 63%


## 5. Pseudo-label loops
Atlas-equal union AP of the three loops, ranked by their saved score (the bars of the headline figure) and ranked by their call set (calls first, in one tie group), which is the ranking the loop sentence of the paper uses.

In [6]:
saved = {m: round(float(ae[m]), 4) for m in ("dup_loop", "nodup_loop", "classweight_loop")}
TC = P.pool_metrics(ASSETS, {m: f"{ASSETS}/predictions/{m}" for m in saved}, pids, call=True)
bycall = {m: round(float(v), 4) for m, v in P.atlas_equal(P.wide(TC, "union_AP"), man).items()}
print("ranked by saved score:", saved); print("ranked by call set   :", bycall)
assert saved == {"dup_loop": 0.0686, "nodup_loop": 0.0720, "classweight_loop": 0.0729}
assert bycall == {"dup_loop": 0.0874, "nodup_loop": 0.0858, "classweight_loop": 0.0870}

ranked by saved score: {'dup_loop': 0.0686, 'nodup_loop': 0.072, 'classweight_loop': 0.0729}
ranked by call set   : {'classweight_loop': 0.087, 'dup_loop': 0.0874, 'nodup_loop': 0.0858}


All assertions above recompute a value from the frozen predictions; the bootstrap interval is the only comparison with a tolerance.